# PennyLane Hamiltonian simulation

Trotterize transverse-field Ising dynamics and compare an expectation-value trajectory.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

This QNode evolves a small Ising system and measures an observable trajectory over time.

In [2]:
times = np.linspace(0.0, 1.2, 13)

def make_qnode(device):
    @qml.qnode(device)
    def evolution(time_value):
        qml.PauliX(0)
        steps = 6
        dt = time_value / steps
        for _ in range(steps):
            qml.IsingZZ(1.1 * dt, wires=[0, 1])
            qml.IsingZZ(1.1 * dt, wires=[1, 2])
            for wire in range(3):
                qml.RX(0.7 * dt, wires=wire)
        return qml.expval(qml.Z(0))
    return evolution

reference_qnode = make_qnode(qml.device("default.qubit", wires=3))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(value) for value in times]))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(value) for value in times]))
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

Every time point must agree with default.qubit.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/11_hamiltonian_simulation.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — observable trajectory atol=3e-6
SDK reference median: 18.792 ms
MettleQ median:       45.988 ms
Timing interpretation: the SDK reference was 2.447x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "observable trajectory atol=3e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_observable_error": 8.914426445905121e-07, "mettleq": [-1.0, -0.9975538849830627, -0.9902538061141968, -0.9782240390777588, -0.9616617560386658, -0.9408407211303711, -0.9161059260368347, -0.8878591060638428, -0.8565614819526672, -0.82271409034729, -0.7868609428405762, -0.7495627999305725, -0.7114009857177734], "reference": [-1.0, -0.9975533999646711, -0.9902542917948216, -0.9782239468098266, -0.9616618196615209, -0.9408416125730157, -0.91610588

## What should you conclude?

Wider and deeper dynamics can cross into GPU-beneficial territory; the small example remains readable and CPU-friendly.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.